# Finding Seed 36 — local RAW RT-DETR R50vd + R101vd

Notebook này chỉ train hai run trên máy Windows hiện tại:

- `raw_rtdetr50` — PekingU RT-DETR-R50vd
- `raw_rtdetr100` — PekingU RT-DETR-R101vd (`100` là tên tương thích report cũ)

Dataset cố định: `benchmakring_transmission_line_defect_model/data/raw_combined` với 18.334 ảnh train và 2.272 ảnh val.

Máy hiện tại có một RTX 4060 Laptop 8 GB, vì vậy cả hai model dùng micro-batch 1, gradient accumulation 16 để giữ effective batch 16. Hai run chạy tuần tự, tự resume từ `last.pt`, và tự hạ AMP sang fp32 cho batch bị NaN theo worker đã ổn định trước đây.

Checkpoint local nằm tại `benchmakring_transmission_line_defect_model/runs/finding_seed/seed36/`. Nếu `USE_CAMBER=True`, notebook khôi phục checkpoint seed36 hiện có và đồng bộ lại lên `stash://bunpmc/projects/finding_seed/seed36/` sau mỗi run dừng hoặc hoàn thành.

Notebook không tự train khi chỉ mở file; training bắt đầu tại cell **Chạy hàng đợi**.


## 0. Cấu hình — bật/tắt từng run

In [2]:
# ===================== CẤU HÌNH LOCAL =====================
SEED = 36
EPOCHS = 50
RUNS = {
    'raw_rtdetr50': True,
    'raw_rtdetr100': True,  # thực tế là R101vd; giữ tên 100 để khớp report cũ
}

GPUS = [0]                    # máy này: RTX 4060 Laptop 8 GB
RT_EFFECTIVE_BATCH = 16       # 1 GPU x batch 1 x accum 16
USE_CAMBER = True             # restore/sync stash seed36; đổi False nếu muốn hoàn toàn offline
POLL_SECONDS = 60
SESSION_BUDGET_HOURS = None   # local: không áp giới hạn 12 giờ kiểu Kaggle
MIN_START_MINUTES = 0
HARD_KILL_GRACE_MINUTES = 15
QUEUE_ORDER = 'as_listed'
# ==========================================================

import time
from pathlib import Path

def find_repo(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'benchmakring_transmission_line_defect_model').is_dir():
            return candidate
    raise FileNotFoundError('Không tìm thấy repository root')

REPO = find_repo(Path.cwd())
BENCHMARK = REPO / 'benchmakring_transmission_line_defect_model'
RAW_DIR = BENCHMARK / 'data/raw_combined'
WORK = BENCHMARK / 'runs/finding_seed'
SEED_DIR = WORK / f'seed{SEED}'
SCRIPTS = WORK / 'seed_scripts'
ARTIFACTS = REPO / 'artifacts/finding_seed' / f'seed{SEED}'
STASH_ROOT = f'stash://bunpmc/projects/finding_seed/seed{SEED}'

SESSION_START = time.time()
DEADLINE_TS = float('inf') if SESSION_BUDGET_HOURS is None else SESSION_START + SESSION_BUDGET_HOURS * 3600
for folder in (SEED_DIR, SCRIPTS, WORK / 'weights', ARTIFACTS / 'data', ARTIFACTS / 'coco'):
    folder.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ['insulator', 'insulator_broken', 'insulator_flashover',
               'bird_nest', 'foreign_object', 'broken_strand']
SPECS = {
    'rtdetr50':  dict(kind='rtdetr', variant='r50vd', batch=1),
    'rtdetr100': dict(kind='rtdetr', variant='r101vd', batch=1),
}
ENABLED = [key for key, enabled in RUNS.items() if enabled]
assert ENABLED and set(ENABLED) <= {'raw_rtdetr50', 'raw_rtdetr100'}
print('Repo:', REPO)
print('Dataset:', RAW_DIR)
print('Runs:', ENABLED)
print('Output:', SEED_DIR)


Repo: D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection
Dataset: D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\benchmakring_transmission_line_defect_model\data\raw_combined
Runs: ['raw_rtdetr50', 'raw_rtdetr100']
Output: D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\benchmakring_transmission_line_defect_model\runs\finding_seed\seed36


## 1. Cài thư viện và kiểm tra GPU

In [3]:
import importlib.metadata, importlib.util, subprocess, sys
import torch

# Checkpoint Camber seed36 được tạo bằng Transformers 4.47.x. Transformers 5.x
# đổi tên nhiều layer RT-DETR (aifi/mlp/o_proj), nên không thể load state_dict cũ.
REQUIRED_TRANSFORMERS = '4.47.1'

def installed_version(package):
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return None

packages = []
if installed_version('transformers') != REQUIRED_TRANSFORMERS:
    packages.append(f'transformers=={REQUIRED_TRANSFORMERS}')
if installed_version('accelerate') != '0.34.2':
    packages.append('accelerate==0.34.2')
required_modules = {
    'pycocotools': 'pycocotools', 'PIL': 'pillow',
    'yaml': 'pyyaml', 'pandas': 'pandas',
}
packages += [package for module, package in required_modules.items() if importlib.util.find_spec(module) is None]
if packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *packages], check=True)
    raise RuntimeError('Đã cài đúng dependency. Restart kernel rồi Run All để loại Transformers cũ khỏi bộ nhớ.')

import transformers, accelerate
assert transformers.__version__ == REQUIRED_TRANSFORMERS, transformers.__version__
assert torch.cuda.is_available(), 'PyTorch không thấy CUDA'
assert max(GPUS) < torch.cuda.device_count(), (GPUS, torch.cuda.device_count())
for index in GPUS:
    props = torch.cuda.get_device_properties(index)
    print(f'GPU {index}: {props.name}, {props.total_memory/2**30:.1f} GiB')
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda,
      '| transformers:', transformers.__version__, '| accelerate:', accelerate.__version__)


GPU 0: NVIDIA GeForce RTX 4060 Laptop GPU, 8.0 GiB
torch: 2.14.1+cu126 | CUDA: 12.6 | transformers: 4.47.1 | accelerate: 0.34.2


## 2. Kiểm tra dataset RAW local và tạo YAML tuyệt đối


In [4]:
import yaml

root = RAW_DIR.resolve()
counts = {f'{kind}/{split}': len([p for p in (root/kind/split).iterdir() if p.is_file() and p.suffix != '.cache'])
          for kind in ('images', 'labels') for split in ('train', 'val')}
assert counts == {'images/train': 18334, 'images/val': 2272,
                  'labels/train': 18334, 'labels/val': 2272}, counts

DATASET_DIRS = {'raw': root}
DATA_YAMLS = {'raw': ARTIFACTS/'data/raw_local.yaml'}
DATA_YAMLS['raw'].write_text(yaml.safe_dump({
    'path': str(root), 'train': 'images/train', 'val': 'images/val',
    'names': dict(enumerate(CLASS_NAMES)),
}, sort_keys=False), encoding='utf-8')
print(root, counts)
print('Generated YAML:', DATA_YAMLS['raw'])


D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\benchmakring_transmission_line_defect_model\data\raw_combined {'images/train': 18334, 'images/val': 2272, 'labels/train': 18334, 'labels/val': 2272}
Generated YAML: D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\artifacts\finding_seed\seed36\data\raw_local.yaml


## 3. Kiểm tra checkpoint local (ưu tiên local, không ghi đè)


In [5]:
import csv, shutil

def epochs_done(run_dir):
    path = Path(run_dir)/'results.csv'
    return max(0, len(path.read_text(encoding='utf-8').strip().splitlines())-1) if path.is_file() else 0

def is_done(run_dir):
    run_dir = Path(run_dir)
    return (run_dir/'done.json').is_file() and epochs_done(run_dir) >= EPOCHS

def has_checkpoint(run_dir):
    return (Path(run_dir)/'weights/last.pt').is_file()

for run_key in ENABLED:
    run_dir = SEED_DIR/run_key
    fake = run_dir/'done.json'
    if fake.is_file() and epochs_done(run_dir) < EPOCHS:
        fake.rename(run_dir/'done_invalid.json')
    print(f'{run_key}: epochs={epochs_done(run_dir)}/{EPOCHS}, last={has_checkpoint(run_dir)}, done={is_done(run_dir)}')


raw_rtdetr50: epochs=16/50, last=True, done=False
raw_rtdetr100: epochs=1/50, last=True, done=False


## 3b. Tùy chọn restore/sync checkpoint bằng Camber Stash qua WSL


In [6]:
import os, subprocess, time

def load_camber_key():
    if os.environ.get('CAMBER_API_KEY'):
        return
    env_file = REPO/'.env'
    if env_file.is_file():
        for line in env_file.read_text(encoding='utf-8').splitlines():
            key, sep, value = line.partition('=')
            if sep and key.strip() == 'CAMBER_API_KEY' and value.strip():
                os.environ['CAMBER_API_KEY'] = value.strip().strip(chr(34)).strip(chr(39)); return
    raise RuntimeError('Thiếu CAMBER_API_KEY trong environment hoặc .env')

def wsl_path(path):
    path = Path(path).resolve()
    return f'/mnt/{path.drive.rstrip(chr(58)).lower()}{path.as_posix().split(chr(58),1)[1]}'

def camber_cmd(*args, check=True):
    if not USE_CAMBER:
        return None
    load_camber_key()
    env = os.environ.copy()
    parts = [part for part in env.get('WSLENV','').split(':') if part]
    if 'CAMBER_API_KEY' not in parts: parts.append('CAMBER_API_KEY')
    env['WSLENV'] = ':'.join(parts)
    converted = [wsl_path(arg) if isinstance(arg, Path) else str(arg) for arg in args]
    command = ['wsl','--','/home/dotru/.camber/bin/camber','stash',*converted]
    proc = subprocess.run(command,env=env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
    if check and proc.returncode:
        raise RuntimeError(proc.stdout[-3000:])
    return proc

def restore_stash_run(run_key):
    run_dir = SEED_DIR/run_key
    if is_done(run_dir) or has_checkpoint(run_dir) or not USE_CAMBER:
        print(f'[LOCAL KEPT] {run_key}: epochs={epochs_done(run_dir)}/{EPOCHS}, last={has_checkpoint(run_dir)}')
        return
    # Tải từng file vào đúng vị trí. Copy cả remote directory có thể tạo thêm
    # một cấp <run_key>/<run_key> khiến worker không nhìn thấy weights/last.pt.
    restored = []
    for relative in ('weights/last.pt','weights/best.pt','results.csv','done.json','exit_status.json','train.log'):
        local = run_dir/relative
        local.parent.mkdir(parents=True,exist_ok=True)
        remote = f'{STASH_ROOT}/{run_key}/{relative}'
        proc = camber_cmd('cp',remote,local,check=False)
        if proc.returncode == 0 and local.is_file():
            restored.append(relative)
    if not has_checkpoint(run_dir):
        raise FileNotFoundError(f'Camber không trả về {run_key}/weights/last.pt; không train mới để tránh mất resume')
    print(f'[RESTORED] {run_key}: epochs={epochs_done(run_dir)}/{EPOCHS}, files={restored}')

def sync_stash_run(run_key):
    if not USE_CAMBER:
        return
    run_dir = SEED_DIR/run_key
    for relative in ('weights/last.pt','weights/best.pt','results.csv','done.json','exit_status.json','train.log'):
        local = run_dir/relative
        if local.is_file():
            camber_cmd('cp',local,f'{STASH_ROOT}/{run_key}/{relative}')
    print(f'[STASH SAVED] {run_key}')

for run_key in ENABLED:
    restore_stash_run(run_key)


[LOCAL KEPT] raw_rtdetr50: epochs=16/50, last=True
[LOCAL KEPT] raw_rtdetr100: epochs=1/50, last=True


## 4. Script train YOLO (chạy trong tiến trình con, 1 GPU)

In [7]:
script = r'''
import argparse, csv, json, os, sys, time
from pathlib import Path

p = argparse.ArgumentParser()
p.add_argument('--run-key', required=True)
p.add_argument('--data', required=True)
p.add_argument('--weights', required=True)
p.add_argument('--batch', type=int, required=True)
p.add_argument('--epochs', type=int, required=True)
p.add_argument('--seed', type=int, required=True)
p.add_argument('--project', required=True)
a = p.parse_args()

import torch
from ultralytics import YOLO, settings
try:
    settings.update({'raytune': False})  # tắt callback Ray Tune (log có cảnh báo get_trial_id)
except Exception:
    pass

DEADLINE = float(os.environ.get('DEADLINE_TS', 'inf'))
run_dir = Path(a.project) / a.run_key
last, best_pt, results = run_dir / 'weights' / 'last.pt', run_dir / 'weights' / 'best.pt', run_dir / 'results.csv'

def n_rows():
    if not results.is_file():
        return 0
    return max(0, len(results.read_text(encoding='utf-8').strip().splitlines()) - 1)

def write_done():
    rows = list(csv.DictReader(results.open(encoding='utf-8')))
    rows = [{k.strip(): v for k, v in r.items()} for r in rows]
    if len(rows) < a.epochs:
        print(f'[{a.run_key}] CHƯA XONG: results.csv mới có {len(rows)}/{a.epochs} epoch — KHÔNG ghi done.json', flush=True)
        sys.exit(3)
    best = max(rows, key=lambda r: float(r['metrics/mAP50(B)']))
    (run_dir / 'done.json').write_text(json.dumps({
        'run_key': a.run_key, 'seed': a.seed, 'epochs': len(rows), 'best_epoch': int(float(best['epoch'])),
        'best_val_mAP50': float(best['metrics/mAP50(B)']), 'best_val_mAP50_95': float(best['metrics/mAP50-95(B)']),
        'completed_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    }, indent=2), encoding='utf-8')
    print(f'[{a.run_key}] COMPLETE ({len(rows)} epoch)', flush=True)

def retarget_checkpoint(path):
    # Ultralytics khi resume lấy project/name/save_dir/data từ chính checkpoint, không lấy từ tham số train().
    # Checkpoint tạo ở đường dẫn cũ (vd /kaggle/working/seed_runs/seed36) sẽ train tiếp và ghi kết quả
    # sang thư mục cũ, còn thư mục hiện tại đứng yên -> trông như "xong" mà không có epoch mới.
    # Ghi đè các đường dẫn này về thư mục hiện tại trước khi resume.
    ck = torch.load(path, map_location='cpu', weights_only=False)
    want = {'project': str(Path(a.project)), 'name': a.run_key, 'data': a.data,
            'save_dir': str(run_dir), 'exist_ok': True}
    changes = []
    def patch(args, where):
        if args is None:
            return
        d = args if isinstance(args, dict) else vars(args)
        for k, v in want.items():
            if k == 'save_dir' and k not in d:
                continue
            if str(d.get(k)) != str(v):
                changes.append(f'{where}.{k}: {d.get(k)} -> {v}')
                d[k] = v
    patch(ck.get('train_args'), 'train_args')
    for key in ('model', 'ema'):
        m = ck.get(key)
        if m is not None and getattr(m, 'args', None) is not None:
            patch(m.args, key)
    if changes:
        tmp = path.with_suffix('.tmp')
        torch.save(ck, tmp)
        os.replace(tmp, path)
        print(f'[{a.run_key}] sửa đường dẫn trong last.pt:\n  ' + '\n  '.join(changes), flush=True)
    return ck

done_rows = n_rows()
if done_rows >= a.epochs and best_pt.is_file():
    print(f'[{a.run_key}] đã đủ {done_rows} epoch; chỉ ghi done.json', flush=True)
    write_done()
    sys.exit(0)

resume = last.is_file()
if resume:
    ck = retarget_checkpoint(last)
    if ck.get('optimizer') is None:
        print(f'[{a.run_key}] LỖI: last.pt không còn optimizer nhưng mới có {done_rows}/{a.epochs} epoch — '
              f'không resume đúng nghĩa được. Cần train lại run này (xóa thư mục run) hoặc xử lý tay.', flush=True)
        sys.exit(4)
    ck_epoch = int(ck.get('epoch', -1)) + 1
    print(f'[{a.run_key}] RESUME từ epoch {ck_epoch + 1} (results.csv có {done_rows} dòng)', flush=True)
    del ck
else:
    print(f'[{a.run_key}] START seed={a.seed} batch={a.batch}', flush=True)

model = YOLO(str(last if resume else a.weights))
clock = {'t': time.time()}

def on_model_save(trainer):
    # Chạy ngay sau khi last.pt đã ghi xong. Thoát cứng ở đây để last.pt còn optimizer
    # (dừng mềm sẽ chạy final_eval -> strip_optimizer -> không resume được).
    if Path(trainer.save_dir).resolve() != run_dir.resolve():
        print(f'[{a.run_key}] LỖI: Ultralytics đang ghi vào {trainer.save_dir}, không phải {run_dir}', flush=True)
        os._exit(5)
    now = time.time()
    epoch_seconds, clock['t'] = now - clock['t'], now
    if trainer.epoch + 1 < trainer.epochs and now + 1.2 * epoch_seconds > DEADLINE:
        print(f'[PAUSE] {a.run_key}: saved epoch {trainer.epoch + 1}, next epoch would exceed the session budget', flush=True)
        os._exit(75)

model.add_callback('on_model_save', on_model_save)
model.train(
    data=a.data, epochs=a.epochs, imgsz=640, batch=a.batch, device=0,
    optimizer='AdamW', lr0=0.001, lrf=0.01, cos_lr=True, warmup_epochs=3.0,
    weight_decay=0.0005, close_mosaic=10, amp=True, seed=a.seed, deterministic=True,
    workers=2, project=a.project, name=a.run_key, exist_ok=True, save=True,
    save_period=-1, plots=True, resume=resume,
)
after = n_rows()
if after <= done_rows and done_rows < a.epochs:
    print(f'[{a.run_key}] LỖI: train() kết thúc nhưng results.csv không thêm epoch nào ({after}/{a.epochs})', flush=True)
    sys.exit(6)
write_done()
'''
(SCRIPTS / 'train_yolo.py').write_text(script.lstrip(), encoding='utf-8')
print('wrote', SCRIPTS / 'train_yolo.py')


wrote D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\benchmakring_transmission_line_defect_model\runs\finding_seed\seed_scripts\train_yolo.py


## 5. Script train RT-DETR PekingU (1 model trên 2 GPU qua torchrun; vẫn chạy được với 1 GPU)


In [8]:
script = r'''
import argparse, csv, json, math, os, random, sys, time, traceback
from pathlib import Path

import numpy as np
import torch
import torch._utils
if not hasattr(torch._utils, '_chunk_or_narrow_cat'):
    def _chunk_or_narrow_cat(tensor, num_chunks, narrow_dim, cat_dim=0):
        return torch.cat(torch.chunk(tensor, num_chunks, dim=narrow_dim), dim=cat_dim)
    torch._utils._chunk_or_narrow_cat = _chunk_or_narrow_cat
import torch.optim as optim
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Sampler

p = argparse.ArgumentParser()
p.add_argument('--run-key', required=True)
p.add_argument('--dataset-dir', required=True)
p.add_argument('--coco-dir', required=True)
p.add_argument('--variant', choices=['r50vd', 'r101vd'], required=True)
p.add_argument('--batch', type=int, required=True, help='micro-batch TRÊN MỖI GPU')
p.add_argument('--accum', type=int, required=True)
p.add_argument('--epochs', type=int, required=True)
p.add_argument('--seed', type=int, required=True)
p.add_argument('--project', required=True)
a = p.parse_args()

# torchrun đặt RANK / LOCAL_RANK / WORLD_SIZE. Chạy thường (1 GPU) thì WORLD = 1.
RANK = int(os.environ.get('RANK', '0'))
LOCAL_RANK = int(os.environ.get('LOCAL_RANK', '0'))
WORLD = int(os.environ.get('WORLD_SIZE', '1'))
MULTI = WORLD > 1
IS_MAIN = RANK == 0

CLASS_NAMES = ['insulator', 'insulator_broken', 'insulator_flashover',
               'bird_nest', 'foreign_object', 'broken_strand']
DEADLINE = float(os.environ.get('DEADLINE_TS', 'inf'))
USE_AMP = (os.environ.get('RTDETR_AMP', '1') == '1'   # RTDETR_AMP=0 -> fp32 hoàn toàn
           and (torch.cuda.is_available() or os.environ.get('RTDETR_FORCE_AMP') == '1'))
WARMUP_STEPS = int(os.environ.get('RTDETR_WARMUP', '2000'))  # warmup tuyến tính theo bước optimizer, như RT-DETR gốc
MAX_BAD_RATIO = 0.05      # dừng nếu >5% micro-batch của một epoch vẫn NaN sau khi đã thử lại fp32
MAX_BAD_OPT_STEPS = 10    # dừng nếu 10 bước optimizer liên tiếp không dùng được
OOM_EXIT, PAUSE_EXIT = 42, 75
repo_id = f'PekingU/rtdetr_{a.variant}'
run_dir = Path(a.project) / a.run_key
wdir = run_dir / 'weights'
wdir.mkdir(parents=True, exist_ok=True)
last_path, best_path, csv_path = wdir / 'last.pt', wdir / 'best.pt', run_dir / 'results.csv'
status_path = run_dir / 'exit_status.json'
device = torch.device(f'cuda:{LOCAL_RANK}' if torch.cuda.is_available() else 'cpu')


def log(msg):
    if IS_MAIN:
        print(msg, flush=True)


def write_status(status, **extra):
    # Notebook đọc file này để biết run dừng vì lý do gì (mã thoát của torchrun không đáng tin).
    tmp = status_path.with_suffix(f'.tmp{RANK}')
    tmp.write_text(json.dumps({'status': status, 'rank': RANK, 'time': time.strftime('%Y-%m-%d %H:%M:%S'), **extra}),
                   encoding='utf-8')
    os.replace(tmp, status_path)


def is_oom(exc):
    oom_types = tuple(t for t in (getattr(torch, 'OutOfMemoryError', None),
                                  getattr(torch.cuda, 'OutOfMemoryError', None)) if t is not None)
    return isinstance(exc, oom_types) or 'out of memory' in str(exc).lower()


def main():
    if MULTI:
        import torch.distributed as dist
        dist.init_process_group(backend='nccl' if torch.cuda.is_available() else 'gloo')
    if torch.cuda.is_available():
        torch.cuda.set_device(device)

    # Cùng seed trên mọi GPU trước khi dựng model -> head mới khởi tạo giống hệt nhau trên các GPU.
    random.seed(a.seed); np.random.seed(a.seed); torch.manual_seed(a.seed); torch.cuda.manual_seed_all(a.seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

    from transformers import RTDetrConfig, RTDetrForObjectDetection

    class NativeProcessor:
        # Tương đương RTDetrImageProcessor của PekingU: resize 640x640 bilinear, /255, không normalize.
        def __call__(self, images, return_tensors='pt'):
            arr = np.asarray(images.convert('RGB').resize((640, 640), Image.BILINEAR), dtype=np.float32)
            return {'pixel_values': torch.from_numpy(arr.transpose(2, 0, 1) / 255.0).unsqueeze(0)}

    try:
        from transformers import RTDetrImageProcessor
        processor = RTDetrImageProcessor.from_pretrained(repo_id)
    except Exception as exc:
        log(f'RTDetrImageProcessor unavailable, using native processor: {exc}')
        processor = NativeProcessor()

    class CocoYolo(Dataset):
        def __init__(self, img_dir, json_path):
            self.img_dir = Path(img_dir)
            coco = json.loads(Path(json_path).read_text(encoding='utf-8'))
            self.images = {im['id']: im for im in coco['images']}
            self.ids = list(self.images)
            self.anns = {i: [] for i in self.ids}
            for ann in coco['annotations']:
                self.anns[ann['image_id']].append(ann)
        def __len__(self):
            return len(self.ids)
        def __getitem__(self, idx):
            info = self.images[self.ids[idx]]
            image = Image.open(self.img_dir / info['file_name']).convert('RGB')
            w, h = info['width'], info['height']
            boxes, classes = [], []
            for ann in self.anns[info['id']]:
                x, y, bw, bh = ann['bbox']
                boxes.append([(x + bw / 2) / w, (y + bh / 2) / h, bw / w, bh / h])
                classes.append(ann['category_id'])
            target = {
                'class_labels': torch.tensor(classes, dtype=torch.long) if classes else torch.zeros((0,), dtype=torch.long),
                'boxes': torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 4), dtype=torch.float32),
            }
            return processor(images=image, return_tensors='pt')['pixel_values'].squeeze(0), target

    class ShardedEpochSampler(Sampler):
        # Thứ tự shuffle cố định theo (seed, epoch) và giống nhau trên mọi GPU; GPU thứ r lấy phần tử r, r+W, r+2W...
        # Danh sách được nối vòng cho chia hết số GPU -> mọi GPU có đúng cùng số batch.
        def __init__(self, n):
            self.n, self.epoch = n, 0
            self.per_rank = math.ceil(n / WORLD)
        def set_epoch(self, epoch):
            self.epoch = epoch
        def __len__(self):
            return self.per_rank
        def __iter__(self):
            g = torch.Generator()
            g.manual_seed(a.seed * 1000 + self.epoch)
            perm = torch.randperm(self.n, generator=g).tolist()
            perm = (perm + perm[:WORLD])[:self.per_rank * WORLD]
            return iter(perm[RANK::WORLD])

    def collate(batch):
        return torch.stack([b[0] for b in batch]), [b[1] for b in batch]

    def worker_init(worker_id):
        s = torch.initial_seed() % 2**32
        np.random.seed(s); random.seed(s)

    ds_dir, coco_dir = Path(a.dataset_dir), Path(a.coco_dir)
    train_ds = CocoYolo(ds_dir / 'images' / 'train', coco_dir / 'train.json')
    val_ds = CocoYolo(ds_dir / 'images' / 'val', coco_dir / 'val.json')
    # Windows spawn không pickle được CocoYolo khai báo local trong main().
    workers = 0 if os.name == 'nt' else min(max(1, (os.cpu_count() or 4) // WORLD), 4)
    train_sampler = ShardedEpochSampler(len(train_ds))
    train_loader = DataLoader(train_ds, batch_size=a.batch, sampler=train_sampler, num_workers=workers,
                              collate_fn=collate, pin_memory=True, worker_init_fn=worker_init, persistent_workers=False)
    val_loader = DataLoader(torch.utils.data.Subset(val_ds, list(range(len(val_ds)))[RANK::WORLD]), batch_size=a.batch,
                            shuffle=False, num_workers=workers, collate_fn=collate, pin_memory=True,
                            worker_init_fn=worker_init)

    config = RTDetrConfig.from_pretrained(repo_id)
    config.num_labels = len(CLASS_NAMES)
    config.id2label = dict(enumerate(CLASS_NAMES))
    config.label2id = {n: i for i, n in enumerate(CLASS_NAMES)}
    model = RTDetrForObjectDetection.from_pretrained(repo_id, config=config, ignore_mismatched_sizes=True).to(device)

    backbone = [p_ for n, p_ in model.named_parameters() if p_.requires_grad and 'backbone' in n]
    others = [p_ for n, p_ in model.named_parameters() if p_.requires_grad and 'backbone' not in n]
    optimizer = optim.AdamW([{'params': backbone, 'lr': 1e-4}, {'params': others, 'lr': 1e-3}], weight_decay=0.0005)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=a.epochs, eta_min=1e-5)
    params = backbone + others
    n_param = sum(p_.numel() for p_ in params)

    # Bộ scale loss cho AMP, cùng quy tắc với torch GradScaler (x2 sau 2000 bước tốt, /2 khi gradient inf/NaN).
    # Tự quản lý để quyết định "bỏ bước" được đưa ra trên gradient ĐÃ gộp -> mọi GPU quyết định giống nhau.
    amp_state = {'scale': 65536.0 if USE_AMP else 1.0, 'good': 0}

    def atomic_save(obj, path):
        tmp = path.with_suffix('.tmp')
        torch.save(obj, tmp)
        os.replace(tmp, path)

    start_epoch, best_val = 0, float('inf')
    if last_path.is_file():
        ck = torch.load(last_path, map_location='cpu', weights_only=False)
        model.load_state_dict(ck['model_state'])
        optimizer.load_state_dict(ck['optimizer_state'])
        scheduler.load_state_dict(ck['scheduler_state'])
        sc = ck.get('scaler_state') or {}
        if USE_AMP and sc.get('scale'):
            amp_state['scale'] = float(sc['scale'])
            amp_state['good'] = int(sc.get('_growth_tracker', 0))
        start_epoch, best_val = ck['epoch'] + 1, ck['best_val_loss']
        del ck
        log(f'[{a.run_key}] RESUME from epoch {start_epoch + 1} | {WORLD} GPU x batch {a.batch} x accum {a.accum} '
            f'= {WORLD * a.batch * a.accum} ảnh / bước optimizer')
    else:
        log(f'[{a.run_key}] START seed={a.seed} | {WORLD} GPU x batch {a.batch} x accum {a.accum} '
            f'= {WORLD * a.batch * a.accum} ảnh / bước optimizer')

    def freeze_bn_stats(m):
        # Giữ running_mean/var của BN (encoder) ở giá trị pretrained; gamma/beta vẫn được train.
        for mod in m.modules():
            if isinstance(mod, torch.nn.modules.batchnorm._BatchNorm):
                mod.eval()

    def to_device(targets):
        return [{k: v.to(device) for k, v in t.items()} for t in targets]

    def forward_loss(pixels, targets, amp):
        # Trả về loss (tensor) hoặc None nếu loss/box dự đoán không hữu hạn.
        try:
            with torch.amp.autocast(device.type, enabled=amp):
                loss = model(pixel_values=pixels, labels=targets).loss.mean()
        except ValueError:  # matcher gặp box dự đoán NaN
            return None
        return loss if bool(torch.isfinite(loss)) else None

    def try_loss(pixels, targets):
        # Thử AMP trước; nếu NaN/inf (tràn fp16) thì chạy lại riêng batch đó ở fp32, chỉ trên GPU gặp lỗi.
        loss = forward_loss(pixels, targets, USE_AMP)
        if loss is not None:
            return loss, False
        if USE_AMP:
            loss = forward_loss(pixels, targets, False)
        return loss, True

    # ---- Đồng bộ nhiều GPU: mỗi GPU giữ một bản model, gradient được CỘNG GỘP ở mỗi bước optimizer. ----
    # Vector gộp = [gradient của mọi tham số | loss_sum, n_good, n_bad, n_fp32].
    N_STAT = 4
    gbuf = torch.zeros(n_param + N_STAT, device=device, dtype=torch.float32)

    def gather_and_reduce(stats):
        offset = 0
        for p_ in params:
            n = p_.numel()
            if p_.grad is None:
                gbuf[offset:offset + n].zero_()
            else:
                gbuf[offset:offset + n].copy_(p_.grad.reshape(-1))
            offset += n
        gbuf[n_param:] = torch.tensor(stats, device=device, dtype=torch.float32)
        if MULTI:
            dist.all_reduce(gbuf, op=dist.ReduceOp.SUM)
        return gbuf[n_param:].tolist()

    def scatter_grads(divisor):
        offset = 0
        for p_ in params:
            n = p_.numel()
            g = gbuf[offset:offset + n].view_as(p_) / divisor
            if p_.grad is None:
                p_.grad = g.clone()
            else:
                p_.grad.copy_(g)
            offset += n

    def reduce_values(values):
        t = torch.tensor(values, device=device, dtype=torch.float64)
        if MULTI:
            dist.all_reduce(t, op=dist.ReduceOp.SUM)
        return t.tolist()

    def main_says(flag):
        # Quyết định của GPU 0 được phát cho mọi GPU (để cùng dừng/cùng chạy tiếp).
        t = torch.tensor([1 if flag else 0], device=device, dtype=torch.int32)
        if MULTI:
            dist.broadcast(t, src=0)
        return bool(t.item())

    def sync_from_main():
        # Chép trọng số + buffer của GPU 0 sang các GPU khác (phòng lệch số học rất nhỏ tích lũy theo thời gian).
        # Trả về độ lệch checksum giữa các GPU TRƯỚC khi chép (0 = các bản model giống hệt nhau).
        if not MULTI:
            return 0.0
        with torch.no_grad():
            c = torch.stack([p_.double().abs().sum() for p_ in params]).sum().reshape(1)
            hi, lo = c.clone(), c.clone()
            dist.all_reduce(hi, op=dist.ReduceOp.MAX)
            dist.all_reduce(lo, op=dist.ReduceOp.MIN)
            for t in list(model.parameters()) + list(model.buffers()):
                if t.numel():
                    dist.broadcast(t, src=0)
        return float((hi - lo).item() / max(abs(hi.item()), 1e-12))

    sync_from_main()  # bảo đảm mọi GPU bắt đầu từ đúng cùng một bộ trọng số
    for epoch in range(start_epoch, a.epochs):
        t0 = time.time()
        train_sampler.set_epoch(epoch)
        # RNG khác nhau giữa các GPU (nhiễu denoising), cố định theo (seed, epoch, GPU) kể cả sau resume.
        rs = a.seed * 100003 + epoch * 101 + RANK
        random.seed(rs); np.random.seed(rs % 2**32); torch.manual_seed(rs); torch.cuda.manual_seed_all(rs)
        model.train()
        freeze_bn_stats(model)
        optimizer.zero_grad(set_to_none=True)
        n_steps = len(train_loader)
        opt_per_epoch = math.ceil(n_steps / a.accum)
        opt_step = epoch * opt_per_epoch           # bước optimizer toàn cục (đúng cả khi resume)
        epoch_lrs = [g['lr'] for g in optimizer.param_groups]  # lr cosine của epoch này
        tot = {'loss': 0.0, 'good': 0.0, 'bad': 0.0, 'fp32': 0.0, 'skipped_opt': 0}
        local = [0.0, 0.0, 0.0, 0.0]               # loss_sum, n_good, n_bad, n_fp32 kể từ lần gộp trước
        consec_bad_opt = 0
        for step, (pixels, targets) in enumerate(train_loader):
            pixels, targets = pixels.to(device, non_blocking=True), to_device(targets)
            loss, retried = try_loss(pixels, targets)
            if loss is None:
                local[2] += 1                      # batch này không đóng góp gradient
            else:
                (loss.float() * (amp_state['scale'] / a.accum)).backward()
                local[0] += float(loss.item()); local[1] += 1; local[3] += 1 if retried else 0
            del loss
            if (step + 1) % a.accum == 0 or step + 1 == n_steps:
                s_loss, s_good, s_bad, s_fp32 = gather_and_reduce(local)
                local = [0.0, 0.0, 0.0, 0.0]
                tot['loss'] += s_loss; tot['good'] += s_good; tot['bad'] += s_bad; tot['fp32'] += s_fp32
                grads_ok = s_good > 0 and bool(torch.isfinite(gbuf[:n_param]).all())
                if grads_ok:
                    scatter_grads(amp_state['scale'] * WORLD)   # bỏ scale AMP + lấy trung bình giữa các GPU
                    warm = min(1.0, (opt_step + 1) / WARMUP_STEPS) if WARMUP_STEPS > 0 else 1.0
                    for g, lr in zip(optimizer.param_groups, epoch_lrs):
                        g['lr'] = lr * warm
                    torch.nn.utils.clip_grad_norm_(params, max_norm=0.1)
                    optimizer.step()
                    consec_bad_opt = 0
                    if USE_AMP:
                        amp_state['good'] += 1
                        if amp_state['good'] >= 2000:
                            amp_state['scale'] *= 2.0; amp_state['good'] = 0
                else:
                    tot['skipped_opt'] += 1; consec_bad_opt += 1
                    if USE_AMP and s_good > 0:      # gradient fp16 tràn -> giảm scale như GradScaler
                        amp_state['scale'] = max(amp_state['scale'] * 0.5, 1.0); amp_state['good'] = 0
                    log(f'  [SKIP] epoch {epoch + 1} step {step + 1}: bỏ 1 bước optimizer '
                        f'(batch tốt {int(s_good)}, batch NaN {int(s_bad)}, scale {amp_state["scale"]:.0f})')
                optimizer.zero_grad(set_to_none=True)
                opt_step += 1
                seen = tot['good'] + tot['bad']
                if consec_bad_opt >= MAX_BAD_OPT_STEPS or (tot['bad'] > 50 and tot['bad'] > MAX_BAD_RATIO * max(seen, 1)):
                    if not all(bool(torch.isfinite(p_).all()) for p_ in params):
                        raise RuntimeError(f'Trọng số đã NaN ở epoch {epoch + 1} step {step + 1}')
                    raise RuntimeError(f'Quá nhiều batch NaN kể cả sau khi thử lại fp32 '
                                       f'({int(tot["bad"])}/{int(seen)}), model đã phân kỳ')
                done_opt = opt_step - epoch * opt_per_epoch
                if done_opt % max(1, 200 // a.accum) == 0:
                    log(f'  epoch {epoch + 1} step {step + 1}/{n_steps} loss {tot["loss"] / max(tot["good"], 1):.4f} '
                        f'lr_head {optimizer.param_groups[1]["lr"]:.2e} fp32_retry {int(tot["fp32"])} '
                        f'nan_skip {int(tot["bad"])} ({(time.time() - t0) / 60:.0f} min)')
        for g, lr in zip(optimizer.param_groups, epoch_lrs):  # trả lr về giá trị cosine trước khi scheduler.step()
            g['lr'] = lr
        scheduler.step()
        drift = sync_from_main()

        model.eval()
        v_sum, v_n, v_bad = 0.0, 0, 0
        with torch.no_grad():
            for pixels, targets in val_loader:
                pixels, targets = pixels.to(device, non_blocking=True), to_device(targets)
                loss, _ = try_loss(pixels, targets)
                if loss is None:
                    v_bad += 1
                else:
                    v_sum += float(loss.item()); v_n += 1
        v_sum, v_n, v_bad = reduce_values([v_sum, v_n, v_bad])
        if v_n == 0:
            raise RuntimeError(f'Toàn bộ batch val đều NaN ở epoch {epoch + 1}')
        train_loss = tot['loss'] / max(tot['good'], 1)
        val_loss = v_sum / v_n
        epoch_seconds = time.time() - t0
        improved = val_loss < best_val
        best_val = min(best_val, val_loss)
        if IS_MAIN:
            if improved:
                atomic_save({'model_state': model.state_dict(), 'epoch': epoch, 'val_loss': val_loss,
                             'variant': a.variant, 'seed': a.seed}, best_path)
            atomic_save({'epoch': epoch, 'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(),
                         'scheduler_state': scheduler.state_dict(),
                         'scaler_state': {'scale': amp_state['scale'], '_growth_tracker': amp_state['good']},
                         'best_val_loss': best_val, 'world_size': WORLD, 'batch': a.batch, 'accum': a.accum}, last_path)
            new_file = not csv_path.is_file()
            with csv_path.open('a', newline='', encoding='utf-8') as f:
                w = csv.writer(f)
                if new_file:
                    w.writerow(['epoch', 'train_loss', 'val_loss', 'lr_backbone', 'lr_head', 'epoch_seconds'])
                w.writerow([epoch + 1, f'{train_loss:.5f}', f'{val_loss:.5f}', f'{optimizer.param_groups[0]["lr"]:.7f}',
                            f'{optimizer.param_groups[1]["lr"]:.7f}', f'{epoch_seconds:.0f}'])
            log(f'[{a.run_key}] epoch {epoch + 1}/{a.epochs} train {train_loss:.4f} val {val_loss:.4f}'
                f'{" *best*" if improved else ""} ({epoch_seconds / 60:.1f} min) | fp32_retry {int(tot["fp32"])} '
                f'nan_skip {int(tot["bad"])} opt_skip {tot["skipped_opt"]} val_nan {int(v_bad)} gpu_drift {drift:.1e}')
        pause = main_says(epoch + 1 < a.epochs and time.time() + 1.2 * epoch_seconds > DEADLINE)
        if pause:
            if IS_MAIN:
                log(f'[PAUSE] {a.run_key}: saved epoch {epoch + 1}, next epoch would exceed the session budget')
                write_status('paused', epoch=epoch + 1)
            return PAUSE_EXIT if not MULTI else 0

    if IS_MAIN:
        best = torch.load(best_path, map_location='cpu', weights_only=True)
        (run_dir / 'done.json').write_text(json.dumps({
            'run_key': a.run_key, 'seed': a.seed, 'epochs': a.epochs, 'best_epoch': best['epoch'] + 1,
            'best_val_loss': best['val_loss'], 'completed_at': time.strftime('%Y-%m-%d %H:%M:%S'),
        }, indent=2), encoding='utf-8')
        write_status('completed')
        log(f'[{a.run_key}] COMPLETE')
    return 0


if __name__ == '__main__':
    try:
        code = main()
    except BaseException as exc:  # noqa: BLE001 - cần bắt cả lỗi CUDA để ghi trạng thái trước khi thoát
        if isinstance(exc, SystemExit):
            raise
        traceback.print_exc()
        sys.stdout.flush(); sys.stderr.flush()
        if is_oom(exc):
            print(f'[OOM] GPU {RANK}: hết VRAM với batch {a.batch}/GPU', flush=True)
            write_status('oom', batch=a.batch)
            os._exit(OOM_EXIT)
        if not status_path.is_file():
            write_status('failed', error=f'{type(exc).__name__}: {exc}'[:500])
        os._exit(1)
    sys.stdout.flush()
    if MULTI:
        try:
            torch.distributed.destroy_process_group()
        except Exception:
            pass
    sys.exit(code)
'''
(SCRIPTS / 'train_rtdetr.py').write_text(script.lstrip(), encoding='utf-8')
print('wrote', SCRIPTS / 'train_rtdetr.py')


wrote D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\benchmakring_transmission_line_defect_model\runs\finding_seed\seed_scripts\train_rtdetr.py


## 6. Tải pretrained PekingU và tạo COCO JSON local


In [9]:
import json
from PIL import Image

pending = [k for k in ENABLED if not is_done(SEED_DIR / k)]
print('Cần chạy:', pending or 'không còn run nào')

# 1) Chỉ tải pretrained weights nếu run chưa có checkpoint resume.
# Với last.pt trong Working, train_yolo.py sẽ tự YOLO(last.pt) + resume=True.
for model_key in sorted({k.split('_', 1)[1] for k in pending}):
    spec = SPECS[model_key]
    if spec['kind'] == 'yolo':
        spec['weights_path'] = str(WORK / 'weights' / spec['weights'])
        related = [k for k in pending if k.split('_', 1)[1] == model_key]
        needs_pretrained = any(not (SEED_DIR / k / 'weights' / 'last.pt').is_file() for k in related)
        if needs_pretrained:
            YOLO(spec['weights'])
        else:
            print(f'  {model_key}: dùng last.pt trong Working, bỏ qua tải pretrained')
    else:
        from huggingface_hub import snapshot_download
        snapshot_download(f"PekingU/rtdetr_{spec['variant']}")

# 2) YOLO -> COCO JSON cho RT-DETR (giống kaggle_train_rtdetr_peking.ipynb: jpg/jpeg/png, bbox pixel xywh).
def yolo_to_coco(ds, split):
    out = ARTIFACTS / 'coco' / ds / f'{split}.json'
    if out.is_file():
        return
    out.parent.mkdir(parents=True, exist_ok=True)
    root = DATASET_DIRS[ds]
    files = sorted(p for p in (root / 'images' / split).iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png'))
    images, anns = [], []
    for img_id, img in enumerate(files):
        with Image.open(img) as im:
            w, h = im.size
        images.append({'id': img_id, 'file_name': img.name, 'width': w, 'height': h})
        lbl = root / 'labels' / split / f'{img.stem}.txt'
        if lbl.is_file():
            for line in lbl.read_text(encoding='utf-8').splitlines():
                parts = line.split()
                if len(parts) >= 5:
                    xc, yc, bw, bh = (float(v) for v in parts[1:5])
                    anns.append({'id': len(anns), 'image_id': img_id, 'category_id': int(parts[0]),
                                 'bbox': [round((xc - bw / 2) * w, 2), round((yc - bh / 2) * h, 2), round(bw * w, 2), round(bh * h, 2)],
                                 'area': round(bw * w * bh * h, 2), 'iscrowd': 0})
    out.write_text(json.dumps({'images': images, 'annotations': anns,
                               'categories': [{'id': i, 'name': n} for i, n in enumerate(CLASS_NAMES)]}), encoding='utf-8')
    print(f'  {out}: {len(images):,} ảnh, {len(anns):,} box')

for ds in sorted({k.split('_', 1)[0] for k in pending if SPECS[k.split('_', 1)[1]]['kind'] == 'rtdetr'}):
    for split in ('train', 'val'):
        yolo_to_coco(ds, split)


Cần chạy: ['raw_rtdetr50', 'raw_rtdetr100']


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

## 7. Chạy tuần tự RAW RT-DETR trên một GPU

Log: `benchmakring_transmission_line_defect_model/runs/finding_seed/seed36/<run>/train.log`. Cell này mới thực sự bắt đầu training.


In [ ]:
import signal, socket

N_GPU = len(GPUS)
SCHED_SLEEP_SECONDS = globals().get('SCHED_SLEEP_SECONDS', 10)   # chu kỳ kiểm tra tiến trình (giây)

def kind_of(run_key):
    return SPECS[run_key.split('_', 1)[1]]['kind']

def gpus_needed(run_key):
    # YOLO: 1 GPU / run. RT-DETR: 1 run chiếm toàn bộ GPU (train 1 model trên 2 GPU cùng lúc).
    return N_GPU if kind_of(run_key) == 'rtdetr' else 1

def rt_batch(run_key):
    # micro-batch TRÊN MỖI GPU. Nếu run từng hết VRAM thì dùng giá trị đã hạ trong batch_override.json.
    spec = SPECS[run_key.split('_', 1)[1]]
    f = SEED_DIR / run_key / 'batch_override.json'
    batch = json.loads(f.read_text())['batch'] if f.is_file() else spec['batch']
    accum = max(1, round(RT_EFFECTIVE_BATCH / (batch * N_GPU)))   # giữ effective batch = 16 như seed 42
    return batch, accum

def free_port():
    with socket.socket() as s:
        s.bind(('127.0.0.1', 0))
        return s.getsockname()[1]

def command(run_key, gpus):
    ds, model_key = run_key.split('_', 1)
    spec = SPECS[model_key]
    if spec['kind'] == 'yolo':
        return [sys.executable, str(SCRIPTS / 'train_yolo.py'), '--run-key', run_key, '--data', str(DATA_YAMLS[ds]),
                '--weights', spec['weights_path'], '--batch', str(spec['batch']), '--epochs', str(EPOCHS),
                '--seed', str(SEED), '--project', str(SEED_DIR)]
    batch, accum = rt_batch(run_key)
    args = [str(SCRIPTS / 'train_rtdetr.py'), '--run-key', run_key, '--dataset-dir', str(DATASET_DIRS[ds]),
            '--coco-dir', str(ARTIFACTS / 'coco' / ds), '--variant', spec['variant'], '--batch', str(batch),
            '--accum', str(accum), '--epochs', str(EPOCHS), '--seed', str(SEED), '--project', str(SEED_DIR)]
    if len(gpus) > 1:   # torchrun: mỗi GPU một tiến trình, cùng train MỘT model
        return [sys.executable, '-m', 'torch.distributed.run', '--nproc_per_node', str(len(gpus)),
                '--master_port', str(free_port()), *args]
    return [sys.executable, *args]

# Thứ tự hàng đợi. YOLO chạy từng cặp trên 2 GPU; RT-DETR cần cả 2 GPU nên chỉ bắt đầu khi cả hai cùng rảnh.
if QUEUE_ORDER == 'yolo_first':
    waiting = sorted(pending, key=lambda k: kind_of(k) == 'rtdetr')
elif QUEUE_ORDER == 'rtdetr_first':
    waiting = sorted(pending, key=lambda k: kind_of(k) != 'rtdetr')
else:
    waiting = list(pending)
print('Thứ tự chạy:', waiting or 'không còn run nào', flush=True)

free_gpus, active, outcomes = list(GPUS), {}, {}

def start(run_key):
    gpus = [free_gpus.pop(0) for _ in range(gpus_needed(run_key))]
    run_dir = SEED_DIR / run_key
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / 'exit_status.json').unlink(missing_ok=True)
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=','.join(map(str, gpus)), DEADLINE_TS=str(DEADLINE_TS),
               PYTHONUNBUFFERED='1')
    log = (run_dir / 'train.log').open('a', encoding='utf-8')
    log.write(f'\n===== session {time.strftime("%Y-%m-%d %H:%M:%S")} on GPU {gpus} =====\n')
    log.flush()
    proc = subprocess.Popen(command(run_key, gpus), stdout=log, stderr=subprocess.STDOUT, env=env, cwd=str(WORK),
                            start_new_session=True)   # nhóm tiến trình riêng để kill được cả các tiến trình con của torchrun
    active[run_key] = dict(proc=proc, gpus=gpus, t0=time.time(), log=log)
    extra = ''
    if kind_of(run_key) == 'rtdetr':
        batch, accum = rt_batch(run_key)
        extra = f' | {len(gpus)} GPU x batch {batch} x accum {accum}'
    print(f'[START] {run_key} trên GPU {gpus}{extra}', flush=True)

def finish(run_key, rc):
    info = active.pop(run_key)
    info['log'].close()
    free_gpus.extend(info['gpus']); free_gpus.sort()
    run_dir = SEED_DIR / run_key
    status = {}
    status_file = run_dir / 'exit_status.json'
    if status_file.is_file():
        try:
            status = json.loads(status_file.read_text(encoding='utf-8'))
        except Exception:
            status = {}
    retry = False
    if rc == 0 and is_done(run_dir):
        outcome = 'completed'
    elif rc == 75 or status.get('status') == 'paused':
        outcome = 'paused'
    elif status.get('status') == 'oom':
        batch, _ = rt_batch(run_key)
        if batch > 1:
            (run_dir / 'batch_override.json').write_text(json.dumps({'batch': batch // 2}), encoding='utf-8')
            outcome, retry = f'hết VRAM ở batch {batch}/GPU -> chạy lại với batch {batch // 2}/GPU', True
        else:
            outcome = 'failed (hết VRAM ngay cả với batch 1/GPU)'
    else:
        outcome = f'failed (exit {rc})'
    outcomes[run_key] = outcome
    print(f'[GPU{info["gpus"]}] {run_key}: {outcome} | epochs={epochs_done(run_dir)}/{EPOCHS}', flush=True)
    if retry:
        waiting.insert(0, run_key)
        return
    # Chỉ sync sau khi tiến trình đã dừng, tránh copy checkpoint lúc đang ghi.
    try:
        sync_stash_run(run_key)
    except Exception as e:
        print(f'[STASH ERROR] {run_key}: {e}', flush=True)
        outcomes[run_key] += ' / STASH SYNC FAILED'
    if outcome.startswith('failed'):
        print('   ' + '\n   '.join((run_dir / 'train.log').read_text(encoding='utf-8', errors='replace').splitlines()[-25:]), flush=True)

last_print = 0.0
while waiting or active:
    for run_key in list(active):
        rc = active[run_key]['proc'].poll()
        if rc is not None:
            finish(run_key, rc)
    can_start = SESSION_BUDGET_HOURS is None or time.time() < DEADLINE_TS - MIN_START_MINUTES * 60
    # Giữ đúng thứ tự: run đứng đầu chưa đủ GPU thì chờ, không cho run phía sau chen lên (tránh RT-DETR bị chờ mãi).
    while can_start and waiting and gpus_needed(waiting[0]) <= len(free_gpus):
        start(waiting.pop(0))
    if not active and (not can_start or not waiting):
        break
    left = float('inf') if SESSION_BUDGET_HOURS is None else (DEADLINE_TS - time.time()) / 60
    if SESSION_BUDGET_HOURS is not None and left < -HARD_KILL_GRACE_MINUTES:
        for run_key, info in active.items():
            print(f'[KILL] {run_key} vượt budget quá {HARD_KILL_GRACE_MINUTES} phút', flush=True)
            try:
                os.killpg(info['proc'].pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
    if time.time() - last_print >= POLL_SECONDS / 2:
        last_print = time.time()
        running = [f'GPU{info["gpus"]}:{rk} ep {epochs_done(SEED_DIR / rk)}/{EPOCHS} ({(time.time() - info["t0"]) / 3600:.1f}h)'
                   for rk, info in active.items()]
        tail = f' | chờ: {waiting}' if waiting else ''
        print(f'{time.strftime("%H:%M")} | ' + (f'còn {left:.0f} phút | ' if SESSION_BUDGET_HOURS is not None else '') + (' | '.join(running) or 'idle') + tail, flush=True)
    time.sleep(SCHED_SLEEP_SECONDS)

if waiting:
    print('Chưa kịp chạy trong phiên này:', waiting, flush=True)
print('Kết quả phiên:', json.dumps(outcomes, indent=2, ensure_ascii=False))


Thứ tự chạy: ['raw_rtdetr50', 'raw_rtdetr100']
[START] raw_rtdetr50 trên GPU [0] | 1 GPU x batch 1 x accum 16
14:28 | GPU[0]:raw_rtdetr50 ep 16/50 (0.0h) | chờ: ['raw_rtdetr100']
14:29 | GPU[0]:raw_rtdetr50 ep 16/50 (0.0h) | chờ: ['raw_rtdetr100']
14:29 | GPU[0]:raw_rtdetr50 ep 16/50 (0.0h) | chờ: ['raw_rtdetr100']
14:30 | GPU[0]:raw_rtdetr50 ep 16/50 (0.0h) | chờ: ['raw_rtdetr100']
14:30 | GPU[0]:raw_rtdetr50 ep 16/50 (0.0h) | chờ: ['raw_rtdetr100']
14:31 | GPU[0]:raw_rtdetr50 ep 16/50 (0.0h) | chờ: ['raw_rtdetr100']
14:31 | GPU[0]:raw_rtdetr50 ep 16/50 (0.1h) | chờ: ['raw_rtdetr100']
14:32 | GPU[0]:raw_rtdetr50 ep 16/50 (0.1h) | chờ: ['raw_rtdetr100']
14:32 | GPU[0]:raw_rtdetr50 ep 16/50 (0.1h) | chờ: ['raw_rtdetr100']
14:33 | GPU[0]:raw_rtdetr50 ep 16/50 (0.1h) | chờ: ['raw_rtdetr100']
14:33 | GPU[0]:raw_rtdetr50 ep 16/50 (0.1h) | chờ: ['raw_rtdetr100']
14:34 | GPU[0]:raw_rtdetr50 ep 16/50 (0.1h) | chờ: ['raw_rtdetr100']
14:34 | GPU[0]:raw_rtdetr50 ep 16/50 (0.1h) | chờ: ['raw_rtdet

## 8. Trạng thái 12 run (chạy lúc nào cũng được)

In [ ]:
import pandas as pd

def summarize(run_key):
    run_dir = SEED_DIR / run_key
    done = json.loads((run_dir / 'done.json').read_text()) if is_done(run_dir) else {}
    row = {'run': run_key, 'enabled': RUNS[run_key],
           'status': 'completed' if done else 'in progress' if epochs_done(run_dir) else 'pending',
           'epochs': f'{epochs_done(run_dir)}/{EPOCHS}'}
    f = run_dir / 'results.csv'
    if f.is_file():
        df = pd.read_csv(f)
        df.columns = [c.strip() for c in df.columns]
        if 'metrics/mAP50(B)' in df:
            i = df['metrics/mAP50(B)'].idxmax()
            row.update(best_epoch=int(df.loc[i, 'epoch']), val_mAP50=round(df.loc[i, 'metrics/mAP50(B)'], 4),
                       val_mAP50_95=round(df.loc[i, 'metrics/mAP50-95(B)'], 4))
        elif 'val_loss' in df:
            i = df['val_loss'].idxmin()
            row.update(best_epoch=int(df.loc[i, 'epoch']), val_loss=round(df.loc[i, 'val_loss'], 4))
    return row

status = pd.DataFrame([summarize(k) for k in ENABLED])
status.to_csv(SEED_DIR / 'status.csv', index=False)
display(status)
if USE_CAMBER:
    camber_cmd('cp', SEED_DIR/'status.csv', STASH_ROOT+'/status.csv')
    print('[STASH SAVED] status.csv')


## 9. Đóng gói best.pt để tải về

In [ ]:
import zipfile

# Gói nhẹ chỉ chứa best.pt + log để tải về đánh giá local (thư mục projects/finding_seed/seed36 đầy đủ vẫn nằm trong Working để resume).
zip_path = WORK / f'seed{SEED}_best_weights.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_STORED) as z:
    for run_key in ENABLED:
        run_dir = SEED_DIR / run_key
        for rel in ('weights/best.pt', 'results.csv', 'args.yaml', 'done.json'):
            if (run_dir / rel).is_file():
                z.write(run_dir / rel, f'seed{SEED}/{run_key}/{rel}')
    z.write(SEED_DIR / 'status.csv', f'seed{SEED}/status.csv')
print(f'{zip_path} ({zip_path.stat().st_size / 2**20:.0f} MB)')
if USE_CAMBER:
    camber_cmd('cp', zip_path, STASH_ROOT+'/'+zip_path.name)
    print('[STASH SAVED] best weights ZIP')
print(f'Tổng dung lượng /kaggle/working: {sum(p.stat().st_size for p in WORK.rglob("*") if p.is_file()) / 2**30:.1f} GB ')


## Sau khi chạy

- Checkpoint resume: `benchmakring_transmission_line_defect_model/runs/finding_seed/seed36/<run>/weights/last.pt`.
- Best checkpoint: cùng thư mục, file `best.pt`.
- COCO JSON chuyển đổi được cache tại `artifacts/finding_seed/seed36/coco/raw/`.
- Hai model chạy tuần tự trên GPU 0; R101 chỉ bắt đầu sau khi R50 hoàn tất hoặc dừng.
- Nếu `USE_CAMBER=True`, từng run được đồng bộ về `stash://bunpmc/projects/finding_seed/seed36/`.
- Chạy lại notebook sẽ resume checkpoint; không xóa thư mục run giữa các phiên.
